Stream2segment offers a simple interface to interact with the database. No need to dig into the complexity of Object-Relational Mapping or SQL syntax, the user just needs to know that:

- Each waveform segment is exposed as a simple Python object called `Segment` with a rich set of attributes and methods

- Many segment attributes can also be used to perform flexible and complex selections (nearly impossible with a classical file system storage) thanks also to a simplified syntax using simple string expressions


## Table of contents

- [Segment (selectable) attributes and methods](#attributes-and-methods)
- [Segments selection](#segments-selection)

## Attributes and methods

The list of all Segment attributes and methods can be printed with few lines of code in your Notebook as shown below (for a list of methods, scroll to the bottom of the table). Included in the list are also attributes of the so-called related objects (e.g. `segment.event`, `segment.station`, `segment.channel`). Attributes are "selectable" in that they can be used to perform powerful cusom selections (see next section).

| Attribute               | Type     | Notes                                                                                                                                   |
|:------------------------|:---------|:----------------------------------------------------------------------------------------------------------------------------------------|
| `id`                    | int      | segment (unique) db id                                                                                                                  |
| `arrival_time`          | datetime | waveform arrival time (usually P-wave arrival time)                                                                                     |
| `noise_window_sec`      | int      | waveform noise duration (rounded to seconds). It is computed as (arrival_time - start_time)                                             |
| `signal_window_sec`     | int      | waveform signal duration (rounded to seconds). It is computed as (end_time - arrival_time)                                              |
| `sample_rate`           | float    | the waveform sample rate (nominal: it is the channel registered sample rate)                                                            |
| `event_distance_deg`    | float    | distance between the segment station and the event, in degrees                                                                          |
| `event_distance_km`     | float    | distance between the segment station and the event, in km, assuming a spherical Earth of radius 6371 km                                 |
| `event_id`              | int      |                                                                                                                                         |
| `event_time`            | datetime |                                                                                                                                         |
| `event_latitude`        | float    |                                                                                                                                         |
| `event_longitude`       | float    |                                                                                                                                         |
| `event_depth_km`        | float    |                                                                                                                                         |
| `event_magnitude_type`  | str      |                                                                                                                                         |
| `event_magnitude`       | float    |                                                                                                                                         |
| `network_code`          | str      |                                                                                                                                         |
| `station_code`          | str      |                                                                                                                                         |
| `location_code`         | str      |                                                                                                                                         |
| `channel_code`          | str      |                                                                                                                                         |
| `band_code`             | str      | first letter of `channel_code`.                                                                                                         |
| `instrument_code`       | str      | second letter of `channel_code`                                                                                                         |
| `orientation_code`      | str      | third letter of `channel_code`                                                                                                          |
| `latitude`              | float    |                                                                                                                                         |
| `longitude`             | float    |                                                                                                                                         |
| `elevation`             | float    |                                                                                                                                         |
| `depth`                 | float    |                                                                                                                                         |
| `azimuth`               | float    |                                                                                                                                         |
| `dip`                   | float    |                                                                                                                                         |
| `channel_id`            | int      | the channel database id, uniquely identified by the tuple (network code, station code, location_code, channel_code, channel_start_time) |
| `station_id`            | int      | the station database id, uniquely identified by the tuple (network_code, station_code, data_webservice_id)                              |
| `data_webservice_id`    | int      | the webservice id providing the waveform data, uniquely identifying a data center (e.g., geofon, caltec)                                |


<!-- | `gap_score_percent` | int: waveform max gap/overlap (in % of the sampling delta). `0`: no gaps/overlaps; `>=100`: gaps; `<=-100`: overlaps. Values between `-50` and `50` are usually considered gap-free |
-->

## Segments selection

The selection of suitable segments for processing (e.g., functions `imap` or `process`, or YAML configuration file for the command `s2s process`) is performed by creating a `dict` mapping one or more Segment attributes to a selection expression for that attribute: 

### Syntax <a name='selexpr_examples'></a>

```python
segments_selection: {
  "[attribute]": "[expression]",
  "[attribute]": "[expression]",
  ...
}
```

Or, in YAML syntax (if you are implementing your own processing config):

```yaml
segments_selection: 
  [attribute]: "[expression]"
  [attribute]: "[expression]"
  ...
```

(the variable name `segments_selection` is mandatory only in a YAML configuration file to be passed to the command `s2s process ...`)

Examples:

1. Not all segments on the database have waveform data. This might happen for several reason (server error, connection error and so on). As such, **in most cases you might probably want to work with segments with waveform data** (`"has_data" : "true"`) **or, to be even more strict, wit valid data**:
```python
{"has_valid_data": "true"}
```
2. To select and work on segments of stations activated in 2017 only:
```python
{"station.start_time": "[2017-01-01, 2018-01-01T00:00:00)"}
```
(brackets denote intervals. Square brackets include end-points, round brackets exclude endpoints)

3. To select segments from specified ids, e.g. 1, 4, 342, 67 (e.g., ids which raised errors during
   a previous run and whose id where logged might need inspection in the GUI):
   segment_select:
```python
{"id": "1 4 342 67"}
```

4. To select segments whose event magnitude is greater than 4.2:
```python
{"event.magnitude": ">4.2"}
```
(the same way work the operators: =, >=, <=, <, !=)

5. To select segments with a particular channel sensor description:
```python
{"channel.sensor_description": "'GURALP CMG-40T-30S'"}
```
(note: for attributes with str values and spaces, we need to quote twice, as otherwise
"GURALP CMG-40T-30S" would match 'GURALP' and 'CMG-40T-30S', but not the whole string.
See list of selectable attributes)